In [1]:
#%pip install --upgrade pip
#%pip install pandas pyarrow numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

In [2]:
# Importing clean dataset
df_listings = pd.read_parquet('../data/clean_listings.parquet', engine='auto')
df_rates = pd.read_parquet('../data/clean_past_rates.parquet', engine='auto')

In [4]:
rates_agg = df_rates.groupby('listing_id').agg(

    # PRICING
    rate_avg_mean           = ('rate_avg', 'mean'),
    rate_avg_median         = ('rate_avg', 'median'),
    rate_avg_std            = ('rate_avg', 'std'),
    booked_rate_avg_mean    = ('booked_rate_avg', 'mean'),   # what guests actually paid
    booked_rate_avg_median  = ('booked_rate_avg', 'median'),
    revpar_mean             = ('revpar', 'mean'),            # revenue per available room
    revpar_std              = ('revpar', 'std'),

    # DEMAND / OCCUPANCY
    occupancy_rate_mean     = ('occupancy_rate', 'mean'),
    occupancy_rate_max      = ('occupancy_rate', 'max'),     # peak demand
    occupancy_rate_std      = ('occupancy_rate', 'std'),     # how volatile demand is
    vacant_days_mean        = ('vacant_days', 'mean'),
    reserved_days_mean      = ('reserved_days', 'mean'),

    # REVENUE
    revenue_mean            = ('revenue', 'mean'),
    revenue_total           = ('revenue', 'sum'),

    # BOOKING
    booking_lead_time_mean  = ('booking_lead_time_avg', 'mean'),  # how far in advance guests book
    length_of_stay_mean     = ('length_of_stay_avg', 'mean'),
    min_night_mean          = ('min_nights_avg', 'mean'),

    # RECORD COUNT
    n_months                = ('date', 'count')

).reset_index()

In [ ]:
peak_months = [6, 7, 8]

peak = df_rates[df_rates['month'].isin(peak_months)]
offpeak = df_rates[~df_rates['month'].isin(peak_months)]

peak_agg = peak.groupby('listing_id').agg(
    peak_rate_avg = ('rate_avg', 'mean'),
    peak_occupancy = ('occupancy_rate', 'mean'),
    peak_revpar = ('revpar', 'mean'),
).reset_index()

offpeak_agg = offpeak.groupby('listing_id').agg(
    offpeak_rate_avg = ('rate_avg', 'mean'),
    offpeak_occupancy = ('occupancy_rate', 'mean'),
    offpeak_revpar = ('revpar', 'mean'),
).reset_index()

# combine everything
rates_agg = rates_agg.merge(peak_agg, on='listing_id', how='left').merge(offpeak_agg, on='listing_id', how='left')

# derive the premium directly
rates_agg['peak_rate_premium'] = rates_agg['peak_rate_avg'] - rates_agg['offpeak_rate_avg']
rates_agg['peak_occupancy_lift'] = rates_agg['peak_occupancy'] - rates_agg['offpeak_occupancy']

In [7]:
listings_extended = df_listings.merge(rates_agg, on='listing_id', how='left')

In [9]:
listings_extended.to_parquet(path='../data/clean_all_data.parquet', engine='pyarrow')